# Test split — scoring and rating the Fable 5.1 sparse_knn row

---
## 1 — Host and working tree


In [1]:
from pathlib import Path

%cd /home/prnamhr/projects/Style-Aware-MT

/home/prnamhr/projects/Style-Aware-MT


In [12]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import getpass
import hashlib
import json
import logging
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone

import yaml

PY = sys.executable
SPLIT = 'test'
OUT = Path('outputs')
RESULTS = Path('results')
ROW = 'fable5_sparse_knn'

# The comparators this row is read out against: the matched Qwen arm, the other commercial
# arm on the same selection, and the baseline all three of them sit above.
COMPARATORS = ['gpt56_sparse_knn', 'sparse_knn', 'knn_fewshot']
READOUT = [ROW, *COMPARATORS]
REFERENCE = 'commercial_haiku'

N_BOOT, ALPHA, SEED = 10000, 0.05, 42
PILOT_N = 20

logging.getLogger('httpx').setLevel(logging.WARNING)
print(f'{datetime.now(timezone.utc):%Y-%m-%d %H:%M}Z, no rater key present')

2026-09-05 07:15Z, no rater key present


---
## 2 — The row being rated


In [7]:
MANIFEST_PATH = OUT / 'test_manifest.json'
MANIFEST = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
EVAL_FILE = Path(MANIFEST['eval_file']['path'])
TEST_SHA = hashlib.sha256(EVAL_FILE.read_bytes()).hexdigest()

HASHES = json.loads(Path('data/splits/hashes.json').read_text(encoding='utf-8'))
assert TEST_SHA == HASHES['hashes']['test.jsonl'] == MANIFEST['eval_file']['sha256']
assert ROW in MANIFEST['rows'], (
    f'{ROW} is not a recorded row; run notebooks/test_fable5_colab.ipynb to completion first'
)

SEGMENTS = [json.loads(x) for x in EVAL_FILE.open(encoding='utf-8') if x.strip()]
TEST_SRC = [r['input'] for r in SEGMENTS]
assert len(SEGMENTS) == MANIFEST['eval_file']['n'] == 1322, len(SEGMENTS)

HEAD = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print(f"{len(MANIFEST['rows'])} rows in the manifest, {len(SEGMENTS)} segments  {TEST_SHA[:16]}")
print(f"{ROW} generated by {MANIFEST['rows'][ROW]['model']} "
      f"effort={MANIFEST['rows'][ROW].get('effort')}, rating at {HEAD[:12]}")

15 rows in the manifest, 1322 segments  3e24e90f55e5e530
fable5_sparse_knn generated by claude-fable-5-1 effort=low, rating at c6502205834f


In [15]:
ROWS = {}
for cond in READOUT + [REFERENCE]:
    path = OUT / f'{cond}_{SPLIT}.jsonl'
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    assert digest == MANIFEST['rows'][cond]['output_sha256'], (
        f'{cond}: {path} is not the file the manifest records; it moved after generation'
    )
    ROWS[cond] = [json.loads(x) for x in path.open(encoding='utf-8') if x.strip()]
    assert len(ROWS[cond]) == len(SEGMENTS), f'{cond}: {len(ROWS[cond])} rows'
    assert [r['input'] for r in ROWS[cond]] == TEST_SRC, f'{cond}: source order differs'
    blank = sum(1 for r in ROWS[cond] if not r['prediction'].strip())
    # A blank prediction is still scored, but it breaks the paired estimators downstream.
    assert blank == 0, f'{cond}: {blank} blank predictions'
    print(f'{cond:<18} {len(ROWS[cond])} rows, aligned, 0 blank  {digest[:12]}')

ROW_SHA = MANIFEST['rows'][ROW]['output_sha256']

fable5_sparse_knn  1322 rows, aligned, 0 blank  19e34d284b64
gpt56_sparse_knn   1322 rows, aligned, 0 blank  4ea9f9f51580
sparse_knn         1322 rows, aligned, 0 blank  8d1b339eda7e
knn_fewshot        1322 rows, aligned, 0 blank  f1d4797ed73e
commercial_haiku   1322 rows, aligned, 0 blank  9758395607ad


---
## 3 — Local scoring

Free, and the same estimators the fourteen rows were scored with. Everything is written to
row-scoped paths: `results/comet_test.json` and the confirmatory ladder are read for the
comparators, never rewritten.

In [3]:
from src.eval.quick import score as surface_score

SURFACE = {c: surface_score(c, OUT, SPLIT) for c in READOUT + [REFERENCE]}
print(f"{'condition':<18} {'chrF':>8} {'BLEU':>8} {'markers/seg':>12}")
for cond in READOUT + [REFERENCE]:
    s = SURFACE[cond]
    print(f"{cond:<18} {s['chrF']:8.2f} {s['BLEU']:8.2f} {s['marker_rate']:12.2f}")
print(f"gold test targets carry {SURFACE[ROW]['ref_marker_rate']:.2f} markers per segment")

condition              chrF     BLEU  markers/seg
fable5_sparse_knn     52.52    28.66         0.80
gpt56_sparse_knn      49.96    25.92         0.78
sparse_knn            40.17    15.87         1.04
knn_fewshot           40.37    15.99         1.01
commercial_haiku      45.98    19.58         0.90
gold test targets carry 0.63 markers per segment


In [4]:
# COMET gets its own interpreter: requirements-comet.txt pins transformers and numpy below
# what the rest of the stack runs on.
COMET_PY = '.venv-comet/bin/python'
if not Path(COMET_PY).exists():
    pip = [COMET_PY, '-m', 'pip', 'install', '-q']
    subprocess.run([PY, '-m', 'venv', '.venv-comet'], check=True)
    subprocess.run([*pip, '--upgrade', 'pip'], check=True)
    subprocess.run([*pip, 'setuptools<81'], check=True)
    subprocess.run([*pip, '-r', 'requirements-comet.txt'], check=True)
subprocess.run([COMET_PY, '-c', 'import comet; print("comet ok")'], check=True)

comet ok


CompletedProcess(args=['.venv-comet/bin/python', '-c', 'import comet; print("comet ok")'], returncode=0)

In [5]:
COMET_PATH = RESULTS / f'comet_{SPLIT}.json'
COMET_ROW_PATH = RESULTS / f'comet_{ROW}_{SPLIT}.json'
subprocess.run([COMET_PY, 'manage.py', 'comet', '--conditions', ROW, '--split', SPLIT,
                '--results_path', str(COMET_ROW_PATH), '--batch_size', '16'], check=True)

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 1006.94it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/home/prnamhr/projects/Style-Aware-MT/.venv-comet/lib/python3.11/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and ver

fable5_sparse_knn COMET 0.7679  (n=1322)
Wrote results/comet_fable5_sparse_knn_test.json


CompletedProcess(args=['.venv-comet/bin/python', 'manage.py', 'comet', '--conditions', 'fable5_sparse_knn', '--split', 'test', '--results_path', 'results/comet_fable5_sparse_knn_test.json', '--batch_size', '16'], returncode=0)

In [8]:
CONFIRMATORY_COMET = json.loads(COMET_PATH.read_text(encoding='utf-8'))
assert ROW not in CONFIRMATORY_COMET, f'{ROW} was scored into the fourteen-row COMET file'
COMET = {**CONFIRMATORY_COMET, **json.loads(COMET_ROW_PATH.read_text(encoding='utf-8'))}

# The row is only comparable to the fourteen if the same checkpoint scored the same segments.
for cond in READOUT + [REFERENCE]:
    rec = COMET[cond]
    assert rec['model'] == COMET[REFERENCE]['model'], (cond, rec['model'])
    assert rec['n'] == len(SEGMENTS), (cond, rec['n'])
    assert rec['sources'] == TEST_SRC, f'{cond}: COMET scored segments in another order'
    assert rec['output_sha256'] == MANIFEST['rows'][cond]['output_sha256'], f'{cond}: unbound'
    print(f"{cond:<18} COMET {rec['system']:.4f}")
print(f"\n{COMET[ROW]['model']} scored all {len(READOUT) + 1} rows on the same "
      f'{len(SEGMENTS)} segments')

fable5_sparse_knn  COMET 0.7679
gpt56_sparse_knn   COMET 0.7610
sparse_knn         COMET 0.6904
knn_fewshot        COMET 0.6905
commercial_haiku   COMET 0.7355

Unbabel/wmt22-comet-da scored all 5 rows on the same 1322 segments


In [9]:
!{PY} manage.py stylometrics --split {SPLIT} --targets-split test \
    --conditions {' '.join(READOUT + [REFERENCE])}

label              n     lex_density  lex_density_sd  ttr     ttr_sd  root_ttr  root_ttr_sd  sent_len_mean  sent_len_mean_sd  sent_len_var  sent_len_var_sd  marker_rate  marker_rate_sd  stylo_dist
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
target:test        1322  0.4663       0.1067          0.8503  0.1148  3.8815    0.9818       23.2291        15.7474           7.7045        80.3696          0.0335       0.0745          0.4499    
fable5_sparse_knn  1322  0.4451       0.1057          0.8294  0.1256  3.8189    0.9595       24.4835        17.0887           5.9818        58.0942          0.0395       0.0764          0.3994    
gpt56_sparse_knn   1322  0.4571       0.1095          0.8454  0.1179  3.8238    0.9849       22.96          15.6102           6.7167        49.0284          0.0393       0.0734          0.3832    
sparse_knn     

In [10]:
STYLO_PATH = RESULTS / f'stylometrics_ci_{ROW}_{SPLIT}.json'
subprocess.run([PY, 'manage.py', 'stylometrics_ci', '--split', SPLIT,
                '--conditions', *READOUT, REFERENCE,
                '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
                '--results_path', str(STYLO_PATH)], check=True)


Register fit of the main conditions  (split=test, n=1322 segments, resamples=10000, seed=42)
stylo_dist = standardized distance to the target-register centroid; lower is better.

rank  condition          stylo_dist  ci95              P(this rank)  modal rank  mean rank
------------------------------------------------------------------------------------------
1     knn_fewshot        0.3423      [0.2977, 0.3938]  0.822         1 (0.822)   1.19     
2     sparse_knn         0.3536      [0.3099, 0.4063]  0.755         2 (0.755)   1.96     
3     gpt56_sparse_knn   0.3832      [0.3393, 0.4339]  0.713         3 (0.713)   3.10     
4     fable5_sparse_knn  0.3994      [0.3551, 0.4504]  0.779         4 (0.779)   3.75     
5     commercial_haiku   0.4719      [0.4227, 0.5266]  1.000         5 (1.000)   5.00     

Signed z per register feature (95% CI; 0 = on target)
condition          lex_density             ttr                      root_ttr                 marker_rate            
-----------

CompletedProcess(args=['/home/prnamhr/projects/Style-Aware-MT/.venv/bin/python', 'manage.py', 'stylometrics_ci', '--split', 'test', '--conditions', 'fable5_sparse_knn', 'gpt56_sparse_knn', 'sparse_knn', 'knn_fewshot', 'commercial_haiku', '--n_resamples', '10000', '--alpha', '0.05', '--seed', '42', '--results_path', 'results/stylometrics_ci_fable5_sparse_knn_test.json'], returncode=0)

In [11]:
LADDER_PATH = RESULTS / f'stylometrics_ci_ladder_{SPLIT}.json'
CONF_LADDER = json.loads(LADDER_PATH.read_text(encoding='utf-8'))
STYLO = json.loads(STYLO_PATH.read_text(encoding='utf-8'))

boot = STYLO['bootstrap']
assert (boot['n_resamples'], boot['seed'], boot['alpha']) == (N_BOOT, SEED, ALPHA), boot
assert boot['n_segments'] == len(SEGMENTS), boot
# A rebuilt centroid would rescale every distance away from the fourteen-row ladder. The
# comparators are scored per condition, so they must come back at their confirmatory values.
assert STYLO['centroid']['fingerprint'] == CONF_LADDER['centroid']['fingerprint'], STYLO['centroid']
for cond in COMPARATORS + [REFERENCE]:
    here, there = STYLO['cells'][cond], CONF_LADDER['cells'][cond]
    assert here['stylo_dist'] == there['stylo_dist'], (cond, here['stylo_dist'])
    assert here['stylo_dist_ci'] == there['stylo_dist_ci'], cond

print(f"centroid {STYLO['centroid']['fingerprint']}, ranked within this read-out only")
for cond in STYLO['ranking']:
    cell = STYLO['cells'][cond]
    lo, hi = cell['stylo_dist_ci']
    print(f"{cond:<18} stylo_dist {cell['stylo_dist']:.4f} [{lo:.4f}, {hi:.4f}]  "
          f"rank {cell['rank']}")

centroid fd5aec8d69454b02, ranked within this read-out only
knn_fewshot        stylo_dist 0.3423 [0.2977, 0.3938]  rank 1
sparse_knn         stylo_dist 0.3536 [0.3099, 0.4063]  rank 2
gpt56_sparse_knn   stylo_dist 0.3832 [0.3393, 0.4339]  rank 3
fable5_sparse_knn  stylo_dist 0.3994 [0.3551, 0.4504]  rank 4
commercial_haiku   stylo_dist 0.4719 [0.4227, 0.5266]  rank 5


In [12]:
STYLO_PAIRS = {}
for rec in STYLO['paired_all']:
    if ROW not in (rec['a'], rec['b']):
        continue
    other = rec['b'] if rec['a'] == ROW else rec['a']
    flip = 1.0 if rec['a'] == ROW else -1.0
    lo, hi = sorted((flip * rec['ci_low'], flip * rec['ci_high']))
    STYLO_PAIRS[other] = {'diff': flip * rec['diff'], 'ci_low': lo, 'ci_high': hi,
                          'p_value': rec['p_value'], 'significant': rec['significant']}

print(f"{'contrast':<34} {'diff':>8} {'95% CI':>21} {'p':>8}  sig")
for other in COMPARATORS + [REFERENCE]:
    m = STYLO_PAIRS[other]
    ci = f"[{m['ci_low']:+.4f}, {m['ci_high']:+.4f}]"
    print(f"{ROW + ' - ' + other:<34} {m['diff']:+8.4f} {ci:>21} {m['p_value']:8.4f}  "
          f"{'*' if m['significant'] else ''}")
print('\nnegative means the row sits closer to the target register than its comparator')

contrast                               diff                95% CI        p  sig
fable5_sparse_knn - gpt56_sparse_knn  +0.0160    [-0.0239, +0.0557]   0.4228  
fable5_sparse_knn - sparse_knn      +0.0451    [-0.0019, +0.0915]   0.0644  
fable5_sparse_knn - knn_fewshot     +0.0565    [+0.0094, +0.1022]   0.0210  *
fable5_sparse_knn - commercial_haiku  -0.0720    [-0.1079, -0.0367]   0.0000  *

negative means the row sits closer to the target register than its comparator


In [13]:
LOCAL_BOOT = {}
for metric in ('chrf', 'bleu'):
    LOCAL_BOOT[metric] = RESULTS / f'bootstrap_{metric}_{ROW}_{SPLIT}.json'
    subprocess.run([PY, 'manage.py', 'bootstrap', '--metric', metric, '--split', SPLIT,
                    '--conditions', *READOUT, '--baseline', 'knn_fewshot',
                    '--pairs', f'{ROW}:sparse_knn', f'{ROW}:gpt56_sparse_knn',
                    '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
                    '--out', str(LOCAL_BOOT[metric])], check=True)

wrote results/bootstrap_chrf_fable5_sparse_knn_test.json

chrf paired bootstrap  (resamples=10000, split=test)
comparison                            n     diff    ci95              p      sig
--------------------------------------------------------------------------------
fable5_sparse_knn - knn_fewshot       1322  12.675  [11.887, 13.479]  0.0    *  
gpt56_sparse_knn - knn_fewshot        1322  9.958   [9.283, 10.620]   0.0    *  
sparse_knn - knn_fewshot              1322  -0.06   [-0.360, 0.238]   0.703     
fable5_sparse_knn - sparse_knn        1322  12.735  [11.946, 13.527]  0.0    *  
fable5_sparse_knn - gpt56_sparse_knn  1322  2.717   [2.097, 3.344]    0.0    *  

* = 95% CI excludes 0 (difference significant at α=0.05)
wrote results/bootstrap_bleu_fable5_sparse_knn_test.json

bleu paired bootstrap  (resamples=10000, split=test)
comparison                            n     diff    ci95              p      sig
------------------------------------------------------------------------

In [14]:
# manage.py bootstrap --metric comet reads results/comet_test.json, which does not carry this
# row, so the same estimator is applied here over the merged segment scores.
from src.eval.bootstrap import paired_bootstrap

COMET_PAIRS = list(dict.fromkeys(
    [(c, 'knn_fewshot') for c in READOUT if c != 'knn_fewshot']
    + [(ROW, c) for c in ('sparse_knn', 'gpt56_sparse_knn')]
))
COMET_COMPARISONS = [
    {'a': a, 'b': b, **paired_bootstrap(COMET[a]['segments'], COMET[b]['segments'],
                                        n_resamples=N_BOOT, alpha=ALPHA, seed=SEED)}
    for a, b in COMET_PAIRS
]
LOCAL_BOOT['comet'] = RESULTS / f'bootstrap_comet_{ROW}_{SPLIT}.json'
LOCAL_BOOT['comet'].write_text(json.dumps(
    {'metric': 'comet', 'judge_tag': None, 'split': SPLIT, 'conditions': READOUT,
     'baseline': 'knn_fewshot', 'adjacent': False, 'n_resamples': N_BOOT, 'alpha': ALPHA,
     'seed': SEED, 'sources': {'comparators': str(COMET_PATH), ROW: str(COMET_ROW_PATH)},
     'comparisons': COMET_COMPARISONS}, indent=2) + '\n', encoding='utf-8')

print(f"{'contrast':<34} {'n':>5} {'diff':>8} {'95% CI':>21} {'p':>8}  sig")
for rec in COMET_COMPARISONS:
    ci = f"[{rec['ci_low']:+.4f}, {rec['ci_high']:+.4f}]"
    print(f"{rec['a'] + ' - ' + rec['b']:<34} {rec['n']:5d} {rec['diff']:+8.4f} {ci:>21} "
          f"{rec['p_value']:8.4f}  {'*' if rec['significant'] else ''}")
print(f"\nwrote {LOCAL_BOOT['comet']}")

contrast                               n     diff                95% CI        p  sig
fable5_sparse_knn - knn_fewshot     1322  +0.0774    [+0.0728, +0.0822]   0.0000  *
gpt56_sparse_knn - knn_fewshot      1322  +0.0706    [+0.0661, +0.0751]   0.0000  *
sparse_knn - knn_fewshot            1322  -0.0001    [-0.0025, +0.0023]   0.9404  
fable5_sparse_knn - sparse_knn      1322  +0.0775    [+0.0726, +0.0824]   0.0000  *
fable5_sparse_knn - gpt56_sparse_knn  1322  +0.0068    [+0.0040, +0.0097]   0.0000  *

wrote results/bootstrap_comet_fable5_sparse_knn_test.json


In [15]:
print(f"{'row':<18} {'COMET':>8} {'stylo':>8} {'chrF':>7} {'BLEU':>7}")
for cond in READOUT + [REFERENCE]:
    print(f"{cond:<18} {COMET[cond]['system']:8.4f} {STYLO['cells'][cond]['stylo_dist']:8.4f} "
          f"{SURFACE[cond]['chrF']:7.2f} {SURFACE[cond]['BLEU']:7.2f}")
print('\nlocal scoring complete: 0 paid calls, $0.00')

row                   COMET    stylo    chrF    BLEU
fable5_sparse_knn    0.7679   0.3994   52.52   28.66
gpt56_sparse_knn     0.7610   0.3832   49.96   25.92
sparse_knn           0.6904   0.3536   40.17   15.87
knn_fewshot          0.6905   0.3423   40.37   15.99
commercial_haiku     0.7355   0.4719   45.98   19.58

local scoring complete: 0 paid calls, $0.00


---
## 4 — The two raters


In [16]:
JUDGE_CFG_A = 'configs/judge_eval.yaml'
JUDGE_CFG_B = 'configs/judge_eval_gpt.yaml'
TAG_B = 'gpt'
MODELS = {'phi_a': 'claude-haiku-4-5', 'phi_b': 'gpt-5.6-terra'}

CFG_A = yaml.safe_load(Path(JUDGE_CFG_A).read_text(encoding='utf-8'))
CFG_B = yaml.safe_load(Path(JUDGE_CFG_B).read_text(encoding='utf-8'))

assert CFG_A['judge']['model'] == MODELS['phi_a'], CFG_A['judge']
assert CFG_A['judge']['thinking'] is False and CFG_A['judge']['temperature'] == 0.0
assert CFG_B['judge']['model'] == MODELS['phi_b'], CFG_B['judge']
assert CFG_B['judge']['reasoning_effort'] == 'none', 'reasoning would reprice the batch line'
assert CFG_B['tag'] == TAG_B and CFG_B['judge']['provider'] == 'openai', CFG_B
assert CFG_A['template_file'] == CFG_B['template_file'] == 'prompts/judge_eval.txt'

RUBRIC = Path('prompts/judge_eval.txt')
RUBRIC_SHA = hashlib.sha256(RUBRIC.read_bytes()).hexdigest()
frozen = json.loads(Path('prompts/hashes.json').read_text(encoding='utf-8'))['templates']
assert RUBRIC_SHA == frozen['judge_eval.txt']['sha256'], 'the evaluation rubric has drifted'
RUBRIC_DIGEST = frozen['judge_eval.txt']['digest']
print(f'rubric {RUBRIC_SHA[:16]} frozen {frozen["judge_eval.txt"]["frozen_on"]}')

rubric ffd6dad41acb0512 frozen 2026-07-05


In [17]:
from src.eval.judge import judge_results_path, judge_segment_dir

JUDGE_PATHS = {'phi_a': judge_results_path(RESULTS, SPLIT),
               'phi_b': judge_results_path(RESULTS, SPLIT, TAG_B)}
SEG_DIRS = {'phi_a': judge_segment_dir(RESULTS, SPLIT),
            'phi_b': judge_segment_dir(RESULTS, SPLIT, TAG_B)}
USAGE_PATHS = {'phi_a': RESULTS / f'judge_{SPLIT}_usage.json',
               'phi_b': RESULTS / f'judge_{TAG_B}_{SPLIT}_usage.json'}

# Captured before the pass so the fourteen can be shown untouched afterwards, and so the
# cumulative ledgers give this pass's spend by difference.
PRIOR = {r: json.loads(p.read_text(encoding='utf-8')) for r, p in JUDGE_PATHS.items()}
PRIOR_LEDGER = {}
for rater, path in USAGE_PATHS.items():
    cum = json.loads(path.read_text(encoding='utf-8'))['cumulative']
    PRIOR_LEDGER[rater] = (cum['cost_usd'], cum['calls'])

for rater in ('phi_a', 'phi_b'):
    table = PRIOR[rater]
    assert ROW not in table, f'{rater} already carries {ROW}; this pass would buy nothing'
    assert len(table) == 14, sorted(table)
    assert {rec['model'] for rec in table.values()} == {MODELS[rater]}, rater
    assert {rec['template_sha256'] for rec in table.values()} == {RUBRIC_DIGEST}, rater
    for cond in COMPARATORS + [REFERENCE]:
        recorded = MANIFEST['rows'][cond]['output_sha256']
        assert table[cond]['output_sha256'] == recorded, (rater, cond)
    meta = json.loads((SEG_DIRS[rater] / '_meta.json').read_text(encoding='utf-8'))
    assert meta['model'] == MODELS[rater] and meta['template_sha256'] == RUBRIC_DIGEST, meta
    assert ROW not in meta['outputs'], f'{rater}: {ROW} already bound in {SEG_DIRS[rater]}'
    print(f'{rater:<6} {MODELS[rater]:<18} 14 conditions, ledger '
          f'${PRIOR_LEDGER[rater][0]:.4f} over {PRIOR_LEDGER[rater][1]} calls')

phi_a  claude-haiku-4-5   14 conditions, ledger $18.6207 over 18508 calls
phi_b  gpt-5.6-terra      14 conditions, ledger $12.8075 over 17258 calls


---
## 5 — Projection


In [18]:
N_CALLS = len(SEGMENTS)
RATES = {r: PRIOR_LEDGER[r][0] / PRIOR_LEDGER[r][1] for r in ('phi_a', 'phi_b')}
PROJECTED = {r: RATES[r] * N_CALLS for r in RATES}
PROJECTED_TOTAL = sum(PROJECTED.values())

for rater in ('phi_a', 'phi_b'):
    print(f'{rater:<6} {N_CALLS} calls at ${RATES[rater]:.3e}  ${PROJECTED[rater]:.2f}')
print(f'{"total":<6} {2 * N_CALLS} calls  ${PROJECTED_TOTAL:.2f}')

phi_a  1322 calls at $1.006e-03  $1.33
phi_b  1322 calls at $7.421e-04  $0.98
total  2644 calls  $2.31


In [19]:
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')
%pip install -q anthropic==0.109.1 openai==2.41.1


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


---
## 6 — Φ_A, the primary rater


In [20]:
r = subprocess.run([PY, 'manage.py', 'judge', '--conditions', ROW, '--split', SPLIT,
                    '--config', JUDGE_CFG_A, '--limit', str(PILOT_N)], check=False)
assert r.returncode == 0, f'phi_a pilot exited {r.returncode}'
PILOT_USAGE = RESULTS / f'judge_{SPLIT}_{ROW}_pilot_usage.json'
shutil.copy(USAGE_PATHS['phi_a'], PILOT_USAGE)

judge claude-haiku-4-5  tag=(none)  template=prompts/judge_eval.txt [ffd6dad41acb0512]
Judging 20 segments for fable5_sparse_knn with claude-haiku-4-5 ...
  fable5_sparse_knn Φ 3.850  (coverage 100%)

pilot only (--limit 20): segment cache written to results/judge_test_segments, results/judge_test.json deliberately NOT written. Re-run without --limit for the full split.
Judge usage: {'calls': 20, 'prompt_tokens': 9423, 'completion_tokens': 1864, 'cost_usd': 0.0187}
Wrote results/judge_test_usage.json  (cumulative $18.64)


PosixPath('results/judge_test_fable5_sparse_knn_pilot_usage.json')

In [21]:
def reprice(usage_path, rate, label):
    """Realized per-call rate from a pilot's session block, against the projected rate."""
    u = json.loads(usage_path.read_text(encoding='utf-8'))
    assert u['priced'], f'{label}: unpriced model, so cost_usd is a floor rather than a bill'
    s = u['session']
    if s['calls'] == 0:
        # A re-run over a complete pilot cache buys nothing, so there is no new rate to read.
        print(f'{label}: 0 calls billed, its segments were already cached; '
              f'the projected ${rate:.3e} stands')
        return rate
    realized = s['cost_usd'] / s['calls']
    print(f"{label}: {s['calls']} calls, {s['prompt_tokens'] / s['calls']:.0f} in / "
          f"{s['completion_tokens'] / s['calls']:.0f} out per call, ${s['cost_usd']:.4f}")
    print(f'  ${realized:.3e}/call against a projected ${rate:.3e}  x{realized / rate:.2f}')
    assert realized <= 1.25 * rate, (
        f'{label}: ${realized:.3e}/call is more than 1.25x the projection; the full pass '
        f'would cost about ${realized * N_CALLS:.2f}'
    )
    return realized


RATE_A_REAL = reprice(USAGE_PATHS['phi_a'], RATES['phi_a'], 'phi_a pilot')
REVISED_A = RATE_A_REAL * N_CALLS
print(f'\nfull phi_a pass reprices to ${REVISED_A:.2f} against ${PROJECTED["phi_a"]:.2f} '
      f'projected, and the pass as a whole to ${REVISED_A + PROJECTED["phi_b"]:.2f}')

phi_a pilot: 20 calls, 471 in / 93 out per call, $0.0187
  $9.350e-04/call against a projected $1.006e-03  x0.93

full phi_a pass reprices to $1.24 against $1.33 projected, and the pass as a whole to $2.22


In [22]:
r = subprocess.run([PY, 'manage.py', 'judge', '--conditions', ROW, '--split', SPLIT,
                    '--config', JUDGE_CFG_A], check=False)
assert r.returncode == 0, f'phi_a exited {r.returncode}'

judge claude-haiku-4-5  tag=(none)  template=prompts/judge_eval.txt [ffd6dad41acb0512]
Judging 1322 segments for fable5_sparse_knn with claude-haiku-4-5 ...
  resuming judge: 20/1322 already scored
  fable5_sparse_knn Φ 3.590  (coverage 100%)
preserved 14 condition(s) not re-scored: afsp_full, afsp_margin, commercial_haiku, gpt56_sparse_knn, knn_fewshot, peft, peft_afsp, peft_knn, random_fewshot, rlsf_w3_0.0, rlsf_w3_2.0, rlsf_w3_6.0, sparse_knn, zeroshot
Wrote results/judge_test.json
Judge usage: {'calls': 1302, 'prompt_tokens': 616169, 'completion_tokens': 134568, 'cost_usd': 1.289}
Wrote results/judge_test_usage.json  (cumulative $19.93)


---
## 7 — Φ_B, the cross-family second rater

The 24h completion window is the batch contract. A timeout here leaves the job running
server-side and a re-run resumes polling the same batch id from the state file.

In [23]:
r = subprocess.run([PY, 'manage.py', 'judge_batch', '--conditions', ROW, '--split', SPLIT,
                    '--config', JUDGE_CFG_B, '--poll_interval', '60'], check=False)
assert r.returncode == 0, f'phi_b exited {r.returncode}'

judge gpt-5.6-terra [batch]  tag=gpt  template=prompts/judge_eval.txt [ffd6dad41acb0512]
Judging 1322 segments for fable5_sparse_knn with gpt-5.6-terra ...
  submitting 1322 requests ...
  submitted batch batch_6a9b2a9a0d248190bdae8855e9abd1a8
  [validating] 0/0 completed
  [validating] 0/0 completed
  [in_progress] 146/1322 completed
  [finalizing] 1322/1322 completed
  [completed] 1322/1322 completed
  batch completed: wrote 1322 segment(s), 15 unscored
  fable5_sparse_knn Φ 4.193  (coverage 99%)
preserved 14 condition(s) not re-scored: afsp_full, afsp_margin, commercial_haiku, gpt56_sparse_knn, knn_fewshot, peft, peft_afsp, peft_knn, random_fewshot, rlsf_w3_0.0, rlsf_w3_2.0, rlsf_w3_6.0, sparse_knn, zeroshot
Wrote results/judge_gpt_test.json
Judge usage: {'calls': 1322, 'prompt_tokens': 538807, 'completion_tokens': 51971, 'cost_usd': 0.8506}
Wrote results/judge_gpt_test_usage.json  (cumulative $13.66)


---
## 8 — What was bought


In [24]:
PHI = {r: json.loads(p.read_text(encoding='utf-8')) for r, p in JUDGE_PATHS.items()}
COVERAGE_MIN = {'phi_a': 0.97, 'phi_b': 0.95}

for rater, table in PHI.items():
    # The fourteen must come through the merge byte-identical: this pass adds a row, it does
    # not re-score the family the confirmatory tests were read off.
    for cond, prior in PRIOR[rater].items():
        assert table[cond] == prior, f'{rater}: {cond} changed under a pass that only added {ROW}'
    rec = table[ROW]
    assert rec['model'] == MODELS[rater], (rater, rec['model'])
    assert rec['n'] == len(SEGMENTS), (rater, rec['n'])
    assert rec['judge_tag'] == (None if rater == 'phi_a' else TAG_B), rec['judge_tag']
    assert rec['template_sha256'] == RUBRIC_DIGEST, (rater, rec['template_sha256'])
    assert rec['output_sha256'] == ROW_SHA, f'{rater}: scored bytes the manifest does not record'
    assert rec['sources'] == TEST_SRC, f'{rater}: scored segments are not the test split'
    assert rec['coverage'] >= COVERAGE_MIN[rater], (rater, rec['coverage'])

print(f"{'condition':<18} {'Phi_A':>7} {'cov':>6} {'Phi_B':>7} {'cov':>6} {'B-A':>7}")
for cond in READOUT + [REFERENCE]:
    a, b = PHI['phi_a'][cond], PHI['phi_b'][cond]
    print(f"{cond:<18} {a['mean']:7.3f} {a['coverage']:6.3f} {b['mean']:7.3f} "
          f"{b['coverage']:6.3f} {b['mean'] - a['mean']:+7.3f}")

condition            Phi_A    cov   Phi_B    cov     B-A
fable5_sparse_knn    3.590  1.000   4.193  0.989  +0.603
gpt56_sparse_knn     3.536  1.000   4.150  0.982  +0.614
sparse_knn           2.692  1.000   3.512  0.976  +0.820
knn_fewshot          2.711  1.000   3.554  0.983  +0.843
commercial_haiku     3.356  1.000   3.862  0.985  +0.506


In [25]:
# The caches' own record of which generation bytes each rater scored, read back independently
# of the results files.
for rater, seg_dir in SEG_DIRS.items():
    meta = json.loads((seg_dir / '_meta.json').read_text(encoding='utf-8'))
    assert meta['outputs'][ROW]['sha256'] == ROW_SHA, rater
    assert meta['outputs'][ROW]['file'] == str(OUT / f'{ROW}_{SPLIT}.jsonl'), meta['outputs'][ROW]
    print(f'{rater}: {len(meta["outputs"])} conditions bound in {seg_dir}/_meta.json')

phi_a: 15 conditions bound in results/judge_test_segments/_meta.json
phi_b: 15 conditions bound in results/judge_gpt_test_segments/_meta.json


In [26]:
SPEND = {}
for rater, usage_path in USAGE_PATHS.items():
    u = json.loads(usage_path.read_text(encoding='utf-8'))
    assert u['priced'], f'{rater}: unpriced, so cost_usd is a floor rather than a bill'
    prior_usd, prior_calls = PRIOR_LEDGER[rater]
    SPEND[rater] = {'model': u['model'],
                    'usd': round(u['cumulative']['cost_usd'] - prior_usd, 4),
                    'calls': u['cumulative']['calls'] - prior_calls,
                    'cumulative_usd': u['cumulative']['cost_usd']}
    print(f"{rater:<6} {u['model']:<18} ${SPEND[rater]['usd']:8.4f} over "
          f"{SPEND[rater]['calls']:6d} calls")

RATER_USD = round(sum(v['usd'] for v in SPEND.values()), 4)
print(f'\nboth raters on {ROW}: ${RATER_USD:.2f} against ${PROJECTED_TOTAL:.2f} projected')

phi_a  claude-haiku-4-5   $  1.3077 over   1322 calls
phi_b  gpt-5.6-terra      $  0.8506 over   1322 calls

both raters on fable5_sparse_knn: $2.16 against $2.31 projected


---
## 9 — Read-out

Free: every estimator below runs on the stored segment scores. All of it is exploratory. The
row is not in the pre-registered family, and it is written to row-scoped paths so the
fourteen-row artefacts the confirmatory tests were read off are not rewritten.

In [27]:
CI_PATHS, BOOT_PATHS = {}, {}
for rater, tag in (('phi_a', None), ('phi_b', TAG_B)):
    stem = f'{tag}_{ROW}' if tag else ROW
    CI_PATHS[rater] = RESULTS / f'judge_ci_{stem}_{SPLIT}.json'
    cmd = [PY, 'manage.py', 'judge_ci', '--split', SPLIT, '--conditions', *READOUT, REFERENCE,
           '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
           '--results_path', str(CI_PATHS[rater])]
    if tag:
        cmd += ['--tag', tag]
    subprocess.run(cmd, check=True)


Judge register fidelity Phi by condition  (split=test, n=1322 segments, resamples=10000, seed=42)
judge: claude-haiku-4-5  [tag (none)]
Phi = mean 1-5 rubric rating against the authorized reference; higher is better.

rank  condition          class      n     Phi     ci95              sd     P(this rank)  modal rank  mean rank
--------------------------------------------------------------------------------------------------------------
1     fable5_sparse_knn  study      1322  3.5900  [3.5454, 3.6339]  0.825  0.992         1 (0.992)   1.01     
2     gpt56_sparse_knn   study      1322  3.5363  [3.4909, 3.5802]  0.815  0.992         2 (0.992)   1.99     
3     commercial_haiku   reference  1322  3.3555  [3.3071, 3.4017]  0.874  1.000         3 (1.000)   3.00     
4     knn_fewshot        study      1322  2.7110  [2.6596, 2.7617]  0.944  0.839         4 (0.839)   4.16     
5     sparse_knn         study      1322  2.6921  [2.6399, 2.7436]  0.958  0.839         5 (0.839)   4.84     

Sco

In [28]:
for rater, tag in (('phi_a', None), ('phi_b', TAG_B)):
    stem = f'{tag}_{ROW}' if tag else ROW
    BOOT_PATHS[rater] = RESULTS / f'bootstrap_judge_{stem}_{SPLIT}.json'
    subprocess.run([PY, 'manage.py', 'bootstrap', '--metric', 'judge', '--split', SPLIT,
                    *(('--judge_tag', tag) if tag else ()),
                    '--conditions', *READOUT, '--baseline', 'knn_fewshot',
                    '--pairs', f'{ROW}:sparse_knn', f'{ROW}:gpt56_sparse_knn',
                    '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
                    '--out', str(BOOT_PATHS[rater])], check=True)

wrote results/bootstrap_judge_fable5_sparse_knn_test.json

judge paired bootstrap  (resamples=10000, split=test)
comparison                            n     diff    ci95             p       sig
--------------------------------------------------------------------------------
fable5_sparse_knn - knn_fewshot       1322  0.879   [0.821, 0.938]   0.0     *  
gpt56_sparse_knn - knn_fewshot        1322  0.825   [0.768, 0.882]   0.0     *  
sparse_knn - knn_fewshot              1322  -0.019  [-0.057, 0.019]  0.3218     
fable5_sparse_knn - sparse_knn        1322  0.898   [0.839, 0.958]   0.0     *  
fable5_sparse_knn - gpt56_sparse_knn  1322  0.054   [0.010, 0.099]   0.0182  *  

* = 95% CI excludes 0 (difference significant at α=0.05)
wrote results/bootstrap_judge_gpt_fable5_sparse_knn_test.json

judge paired bootstrap  (resamples=10000, split=test)
comparison                            n     diff    ci95             p       sig
----------------------------------------------------------------

In [29]:
AGREE_PATH = RESULTS / f'judge_agreement_{TAG_B}_{ROW}_{SPLIT}.json'
subprocess.run([PY, 'manage.py', 'judge_agreement', '--split', SPLIT, '--conditions', *READOUT,
                '--reference', REFERENCE, '--tag_b', TAG_B, '--n_resamples', str(N_BOOT),
                '--alpha', str(ALPHA), '--seed', str(SEED),
                '--results_path', str(AGREE_PATH)], check=True)


Judge-judge agreement  (split=test, resamples=10000, seed=42, 95% percentile CIs)
  judge A: claude-haiku-4-5  [tag (none)]
  judge B: gpt-5.6-terra  [tag gpt]
  same frozen rubric verified by digest: True

Coverage (segments parsed by each rater)
condition          n_total  n_a   n_b   n_both
----------------------------------------------
fable5_sparse_knn  1322     1322  1307  1307  
gpt56_sparse_knn   1322     1322  1298  1298  
sparse_knn         1322     1322  1290  1290  
knn_fewshot        1322     1322  1300  1300  
commercial_haiku   1322     1322  1302  1302  

Rater agreement -- study_only
condition          n     Phi_A  Phi_B  A-B     ci95              qwk     qwk_ci            rho     exact  adj  
---------------------------------------------------------------------------------------------------------------
fable5_sparse_knn  1307  3.590  4.193  -0.603  [-0.640, -0.565]  +0.502  [+0.462, +0.538]  +0.592  40.6%  93.0%
gpt56_sparse_knn   1298  3.539  4.150  -0.611  [-0.649,

CompletedProcess(args=['/home/prnamhr/projects/Style-Aware-MT/.venv/bin/python', 'manage.py', 'judge_agreement', '--split', 'test', '--conditions', 'fable5_sparse_knn', 'gpt56_sparse_knn', 'sparse_knn', 'knn_fewshot', '--reference', 'commercial_haiku', '--tag_b', 'gpt', '--n_resamples', '10000', '--alpha', '0.05', '--seed', '42', '--results_path', 'results/judge_agreement_gpt_fable5_sparse_knn_test.json'], returncode=0)

In [30]:
AGREEMENT = json.loads(AGREE_PATH.read_text(encoding='utf-8'))
assert AGREEMENT['judges']['template_verified'], AGREEMENT['judges']
row_agree = AGREEMENT['rater_agreement']['study_only']['per_condition'][ROW]
print(f"{ROW}: n={row_agree['n']}  qwk={row_agree['qwk']['kappa']:+.3f}  "
      f"rho={row_agree['spearman']['rho']:+.3f}  exact={row_agree['exact_agreement']:.1%}  "
      f"adjacent={row_agree['adjacent_agreement']:.1%}")
print(f"severity offset A-B = {row_agree['offset']['diff']:+.3f} "
      f"[{row_agree['offset']['ci_low']:+.3f}, {row_agree['offset']['ci_high']:+.3f}]")
ordering = AGREEMENT['condition_ordering']['study_only']
print(f"A: {' > '.join(ordering['ranking_a'])}")
print(f"B: {' > '.join(ordering['ranking_b'])}")
print(f"identical ranking: {ordering['identical_ranking']}")

fable5_sparse_knn: n=1307  qwk=+0.502  rho=+0.592  exact=40.6%  adjacent=93.0%
severity offset A-B = -0.603 [-0.640, -0.565]
A: fable5_sparse_knn > gpt56_sparse_knn > knn_fewshot > sparse_knn
B: fable5_sparse_knn > gpt56_sparse_knn > knn_fewshot > sparse_knn
identical ranking: True


In [31]:
# The contrasts this row exists to make. They are not in judge_agreement's hardcoded families,
# so they are built here, with Holm applied inside this family of three per rater.
from src.eval.judge_agreement import _load_pair, contrast_replication

LOADED = {}
for c in READOUT:
    pair = _load_pair(OUT, SPLIT, c, SEG_DIRS['phi_a'], SEG_DIRS['phi_b'])
    assert pair is not None, f'{c}: one of the raters has no segment cache for it'
    LOADED[c] = pair

CONTRASTS = contrast_replication(
    LOADED, [(ROW, c) for c in COMPARATORS],
    n_resamples=N_BOOT, seed=SEED, alpha=ALPHA,
)

def _ci(rec):
    return f"[{rec['ci_low']:+.3f}, {rec['ci_high']:+.3f}]"


print(f"{'contrast':<34} {'n':>5} {'dA':>7} {'ciA':>18} {'dB':>7} {'ciB':>18}  sign  verdict")
for name, m in CONTRASTS['contrasts'].items():
    a, b = m['judge_a'], m['judge_b']
    verdict = ('both separate' if m['both_separate']
               else 'neither separates' if m['neither_separates'] else 'RATER-DEPENDENT')
    print(f"{name:<34} {m['n']:5d} {a['diff']:+7.3f} {_ci(a):>18} "
          f"{b['diff']:+7.3f} {_ci(b):>18}  "
          f"{'same' if m['same_sign'] else 'FLIP'}  {verdict}")
print('\na RATER-DEPENDENT contrast is not reportable without naming the rater it depends on')

contrast                               n      dA                ciA      dB                ciB  sign  verdict
fable5_sparse_knn - gpt56_sparse_knn  1284  +0.045   [+0.001, +0.090]  +0.037   [-0.004, +0.079]  same  RATER-DEPENDENT
fable5_sparse_knn - sparse_knn      1275  +0.893   [+0.832, +0.955]  +0.685   [+0.630, +0.742]  same  both separate
fable5_sparse_knn - knn_fewshot     1286  +0.876   [+0.816, +0.937]  +0.646   [+0.593, +0.700]  same  both separate

a RATER-DEPENDENT contrast is not reportable without naming the rater it depends on


---
## 10 — Seal


In [32]:
LOCAL_WRITTEN = [COMET_ROW_PATH, STYLO_PATH, *LOCAL_BOOT.values()]
WRITTEN = [*LOCAL_WRITTEN, *JUDGE_PATHS.values(), *USAGE_PATHS.values(), PILOT_USAGE,
           *CI_PATHS.values(), *BOOT_PATHS.values(), AGREE_PATH,
           *(SEG_DIRS[r] / f'{ROW}.jsonl' for r in SEG_DIRS)]
for p in WRITTEN:
    assert p.exists(), p
    print(f'{str(p):<56} {p.stat().st_size / 1024:8.1f} KiB')

# The fourteen-row local artefacts were read for the comparators, not written to.
assert json.loads(COMET_PATH.read_text(encoding='utf-8')) == CONFIRMATORY_COMET, COMET_PATH
assert json.loads(LADDER_PATH.read_text(encoding='utf-8')) == CONF_LADDER, LADDER_PATH

# Nothing scored here may have moved a generation or the split.
for cond in READOUT + [REFERENCE]:
    path = OUT / f'{cond}_{SPLIT}.jsonl'
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST['rows'][cond]['output_sha256']
assert hashlib.sha256(EVAL_FILE.read_bytes()).hexdigest() == TEST_SHA, 'test.jsonl changed'
assert json.loads(MANIFEST_PATH.read_text(encoding='utf-8')) == MANIFEST, 'the manifest moved'

dirty = subprocess.run(['git', 'status', '--porcelain', 'configs', 'outputs', 'results', 'data'],
                       capture_output=True, text=True).stdout.splitlines()
unexpected = [line for line in dirty if 'test' not in line]
assert not unexpected, unexpected
print(f'\n{ROW} scored locally and rated by both raters, no prior row rewritten')

results/judge_test.json                                    8231.5 KiB
results/judge_gpt_test.json                                8233.0 KiB
results/judge_test_usage.json                                 0.4 KiB
results/judge_gpt_test_usage.json                             0.4 KiB
results/judge_test_fable5_sparse_knn_pilot_usage.json         0.4 KiB
results/judge_ci_fable5_sparse_knn_test.json                  5.5 KiB
results/judge_ci_gpt_fable5_sparse_knn_test.json              5.5 KiB
results/bootstrap_judge_fable5_sparse_knn_test.json           1.8 KiB
results/bootstrap_judge_gpt_fable5_sparse_knn_test.json       1.8 KiB
results/judge_agreement_gpt_fable5_sparse_knn_test.json      17.5 KiB
results/judge_test_segments/fable5_sparse_knn.jsonl         222.5 KiB
results/judge_gpt_test_segments/fable5_sparse_knn.jsonl     222.5 KiB


AssertionError: ['?? configs/commercial_fable5_sparse_knn.yaml']

In [33]:
import platform

import sacrebleu

RATING_PATH = RESULTS / f'judge_{ROW}_{SPLIT}_manifest.json'
RATING = {
    'split': SPLIT,
    'commit': HEAD,
    'row': ROW,
    'generation': {'manifest': str(MANIFEST_PATH), 'output_sha256': ROW_SHA,
                   'model': MANIFEST['rows'][ROW]['model'],
                   'runbook': 'notebooks/test_fable5_colab.ipynb'},
    'raters': {'phi_a': {'model': MODELS['phi_a'], 'config': JUDGE_CFG_A, 'tag': None,
                         'transport': 'sync'},
               'phi_b': {'model': MODELS['phi_b'], 'config': JUDGE_CFG_B, 'tag': TAG_B,
                         'transport': 'batch'},
               'rubric': {'file': str(RUBRIC), 'sha256': RUBRIC_SHA, 'digest': RUBRIC_DIGEST}},
    'phi': {r: {k: PHI[r][ROW][k] for k in ('n', 'mean', 'coverage')} for r in PHI},
    'local': {'surface': {c: {k: SURFACE[c][k] for k in ('chrF', 'BLEU', 'marker_rate')}
                          for c in READOUT + [REFERENCE]},
              'comet': {c: COMET[c]['system'] for c in READOUT + [REFERENCE]},
              'stylo_dist': {c: STYLO['cells'][c]['stylo_dist']
                             for c in READOUT + [REFERENCE]},
              'centroid': STYLO['centroid'],
              'paths': {'comet': str(COMET_ROW_PATH), 'stylometrics_ci': str(STYLO_PATH),
                        **{m: str(p) for m, p in LOCAL_BOOT.items()}}},
    'estimators': {'n_resamples': N_BOOT, 'alpha': ALPHA, 'seed': SEED, 'paired': True},
    'evidence_class': 'exploratory; this row is not in the pre-registered confirmatory family',
    'comparators': COMPARATORS,
    'contrasts': CONTRASTS,
    'spend': {'projected_usd': round(PROJECTED_TOTAL, 4),
              'actual_usd': RATER_USD, 'per_rater': SPEND,
              'separate_from': 'the 2026-08-26 two-rater authorization of $35'},
    'artifacts': {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in WRITTEN},
    'versions': {'python': platform.python_version(), 'sacrebleu': sacrebleu.__version__,
                 'comet_model': COMET[ROW]['model']},
}
RATING_PATH.write_text(json.dumps(RATING, indent=2) + '\n', encoding='utf-8')
print(f'Wrote {RATING_PATH}')

Wrote results/judge_fable5_sparse_knn_test_manifest.json


In [34]:
!tar -czf test_fable5_rating.tar.gz \
    results/judge_test.json results/judge_gpt_test.json \
    results/judge_test_usage.json results/judge_gpt_test_usage.json \
    results/judge_test_fable5_sparse_knn_pilot_usage.json \
    results/judge_ci_fable5_sparse_knn_test.json results/judge_ci_gpt_fable5_sparse_knn_test.json \
    results/bootstrap_judge_fable5_sparse_knn_test.json \
    results/bootstrap_judge_gpt_fable5_sparse_knn_test.json \
    results/judge_agreement_gpt_fable5_sparse_knn_test.json \
    results/comet_fable5_sparse_knn_test.json \
    results/stylometrics_ci_fable5_sparse_knn_test.json \
    results/bootstrap_chrf_fable5_sparse_knn_test.json \
    results/bootstrap_bleu_fable5_sparse_knn_test.json \
    results/bootstrap_comet_fable5_sparse_knn_test.json \
    results/judge_fable5_sparse_knn_test_manifest.json \
    results/judge_test_segments/fable5_sparse_knn.jsonl \
    results/judge_gpt_test_segments/fable5_sparse_knn.jsonl
!ls -la test_fable5_rating.tar.gz
!git status --short results

-rw-r--r-- 1 prnamhr prnamhr 2236385 Sep  4 23:37 test_fable5_rating.tar.gz
 M results/judge_gpt_test.json
 M results/judge_gpt_test_segments/_meta.json
 M results/judge_gpt_test_usage.json
 M results/judge_test.json
 M results/judge_test_segments/_meta.json
 M results/judge_test_usage.json
?? results/bootstrap_judge_fable5_sparse_knn_test.json
?? results/bootstrap_judge_gpt_fable5_sparse_knn_test.json
?? results/judge_agreement_gpt_fable5_sparse_knn_test.json
?? results/judge_ci_fable5_sparse_knn_test.json
?? results/judge_ci_gpt_fable5_sparse_knn_test.json
?? results/judge_fable5_sparse_knn_test_manifest.json
?? results/judge_gpt_test_batch/fable5_sparse_knn_0_input.jsonl
?? results/judge_gpt_test_segments/fable5_sparse_knn.jsonl
?? results/judge_test_fable5_sparse_knn_pilot_usage.json
?? results/judge_test_segments/fable5_sparse_knn.jsonl
